In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc
import torch
from torch.utils.data import DataLoader
from scgenemae.data.dataset import SingleCellDataset
from scgenemae.models.mae import SingleCellMAE
from scgenemae.training.evaluate import extract_embeddings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
data_path = "data/processed/train_cells.h5ad"
dataset = SingleCellDataset(data_path, mask_ratio=0.0)
dataloader = DataLoader(dataset, batch_size=4, shuffle=False)

model = SingleCellMAE(num_genes=dataset.num_genes).to(device)
checkpoint_path = "checkpoints/scgenemae_pretrained.pt"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))

embeddings, _ = extract_embeddings(model, dataloader, device)
print(f"Extracted embeddings shape: {embeddings.shape}")

In [ ]:
adata_emb = ad.AnnData(X=embeddings)
n_neighbors = min(5, max(2, len(embeddings) - 1))

sc.pp.neighbors(adata_emb, n_neighbors=n_neighbors, use_rep='X')
sc.tl.umap(adata_emb)
sc.pl.umap(adata_emb, title="scGeneMAE Pretrained Cell Embeddings")